# artificial_peps Top 克隆提取

这个 notebook 用于：

- 支持多个输入路径，递归查找路径下的 `artificial_peps` 目录
- 读取 `TRA`、`TRB`、`TRG`、`TRD` 链的 `csv` / `csv.gz` 文件
- 按 `copy` 降序提取每条链的 Top N 克隆
- 输出单链表格，以及同一样本的 `TRA/TRB`、`TRG/TRD` 配对表格

输出字段统一为：`index`, `Chain`, `CDR3(pep)`, `joinedSeq`, `V`, `D`, `J`, `C`, `copy`


In [ ]:
from pathlib import Path

# 可修改配置
INPUT_ROOTS = [
    "/colddata/data_shared260414/To_ZQY/260423_lu/ARP_20250118_reports/mouse ab/SHMYS_LLM_03",
    "/colddata/data_shared260414/To_ZQY/260423_lu/SHMYS_DXP_3",
]

TOP_N = 10
SEARCH_DIR_NAME = "artificial_peps"
OUTPUT_FOLDER_NAME = "artificial_peps_top_clones"

PAIRINGS = [("TRA", "TRB"), ("TRG", "TRD")]

PAIRED = False

if not INPUT_ROOTS:
    print("请先在 INPUT_ROOTS 中填写一个或多个输入路径。")
else:
    print("当前输入路径：")
    for root in INPUT_ROOTS:
        print(f"- {root}")
    print(f"Top N = {TOP_N}")


CHAIN_LABEL_MAP = {
    "TRA": "Alpha",
    "TRB": "Beta",
    "TRG": "Gamma",
    "TRD": "Delta",
}


当前输入路径：
- /colddata/data_shared260414/To_ZQY/260423_lu/ARP_20250118_reports/mouse ab/SHMYS_LLM_03
- /colddata/data_shared260414/To_ZQY/260423_lu/SHMYS_DXP_3
Top N = 10


In [6]:
import re
from typing import Dict, List, Optional, Tuple

import pandas as pd

OUTPUT_COLUMNS = [
    "index",
    "Chain",
    "CDR3(pep)",
    "joinedSeq",
    "V",
    "D",
    "J",
    "C",
    "copy",
]

COLUMN_ALIASES = {
    "CDR3(pep)": [
        "CDR3(pep)",
        "cdr3(pep)",
        "CDR3_pep",
        "cdr3_pep",
        "CDR3pep",
        "AA",
        "aaSeq",
    ],
    "joinedSeq": ["joinedSeq", "JoinedSeq", "joined_seq", "ntSeq", "NT", "nSeq"],
    "V": ["V", "v", "bestVGene", "Vgene", "vGene"],
    "D": ["D", "d", "bestDGene", "Dgene", "dGene"],
    "J": ["J", "j", "bestJGene", "Jgene", "jGene"],
    "C": ["C", "c", "bestCGene", "Cgene", "cGene"],
    "copy": ["copy", "Copy", "count", "Count", "cloneCount", "readCount", "reads"],
}

FILE_PATTERN = re.compile(
    r"^(?P<sample>.+?)__(?P<chain>TRA|TRB|TRG|TRD)\.csv(?:\.gz)?$", re.IGNORECASE
)


def normalize_filename(file_name: str) -> str:
    return file_name.replace(" ", "")


def parse_chain_file(file_path: Path) -> Optional[Tuple[str, str]]:
    normalized_name = normalize_filename(file_path.name)
    match = FILE_PATTERN.match(normalized_name)
    if not match:
        return None
    sample = match.group("sample")
    chain = match.group("chain").upper()
    return sample, chain


def discover_artificial_peps(root: Path) -> List[Path]:
    return sorted(path for path in root.rglob(SEARCH_DIR_NAME) if path.is_dir())


def find_matching_column(df: pd.DataFrame, logical_name: str) -> Optional[str]:
    candidates = COLUMN_ALIASES[logical_name]
    lower_map = {str(column).strip().lower(): column for column in df.columns}

    for candidate in candidates:
        if candidate in df.columns:
            return candidate
        lower_candidate = candidate.strip().lower()
        if lower_candidate in lower_map:
            return lower_map[lower_candidate]
    return None


def sanitize_sheet_frame(df: pd.DataFrame, chain: str, top_n: int) -> pd.DataFrame:
    column_map: Dict[str, Optional[str]] = {
        logical_name: find_matching_column(df, logical_name)
        for logical_name in COLUMN_ALIASES
    }

    missing_required = [
        name for name in ["CDR3(pep)", "copy"] if column_map[name] is None
    ]
    if missing_required:
        raise ValueError(
            f"缺少必要字段: {missing_required}; 实际字段为: {list(df.columns)}"
        )

    output = pd.DataFrame(index=df.index)
    output["Chain"] = CHAIN_LABEL_MAP.get(chain, chain)
    output["CDR3(pep)"] = df[column_map["CDR3(pep)"]].fillna("").astype(str).str.strip()

    for field in ["joinedSeq", "V", "D", "J", "C"]:
        matched_column = column_map[field]
        if matched_column is None:
            output[field] = ""
        else:
            output[field] = df[matched_column].fillna("").astype(str).str.strip()

    copy_series = (
        df[column_map["copy"]]
        .fillna("")
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    output["copy"] = pd.to_numeric(copy_series, errors="coerce")
    output = output.dropna(subset=["copy"])
    output = output[output["CDR3(pep)"] != ""]
    output = output.sort_values(
        by=["copy", "CDR3(pep)"], ascending=[False, True], kind="mergesort"
    )
    output = output.head(top_n).reset_index(drop=True)
    output.insert(0, "index", range(1, len(output) + 1))
    output = output[OUTPUT_COLUMNS]
    return output


def load_chain_top_table(file_path: Path, chain: str, top_n: int) -> pd.DataFrame:
    df = pd.read_csv(file_path, compression="infer")
    return sanitize_sheet_frame(df, chain=chain, top_n=top_n)


def build_pair_table(left_df: pd.DataFrame, right_df: pd.DataFrame) -> pd.DataFrame:
    paired_rows = []
    pair_count = max(len(left_df), len(right_df))

    for idx in range(pair_count):
        pair_index = idx + 1
        if idx < len(left_df):
            row = left_df.iloc[idx].to_dict()
            row["index"] = pair_index
            paired_rows.append(row)
        if idx < len(right_df):
            row = right_df.iloc[idx].to_dict()
            row["index"] = pair_index
            paired_rows.append(row)

    return pd.DataFrame(paired_rows, columns=OUTPUT_COLUMNS)


def make_output_bucket(root: Path, artificial_dir: Path) -> Path:
    relative_dir = artificial_dir.relative_to(root)
    bucket_name = "__".join(relative_dir.parts)
    output_dir = root / OUTPUT_FOLDER_NAME / bucket_name
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def collect_chain_files(artificial_dir: Path) -> Dict[str, Dict[str, Path]]:
    sample_chain_files: Dict[str, Dict[str, Path]] = {}
    for file_path in sorted(artificial_dir.iterdir()):
        if not file_path.is_file():
            continue
        parsed = parse_chain_file(file_path)
        if parsed is None:
            continue
        sample, chain = parsed
        sample_chain_files.setdefault(sample, {})[chain] = file_path
    return sample_chain_files


In [ ]:
def process_root(root_path: str, top_n: int = TOP_N) -> pd.DataFrame:
    root = Path(root_path)
    if not root.exists():
        raise FileNotFoundError(f"输入路径不存在: {root}")

    artificial_dirs = discover_artificial_peps(root)
    if not artificial_dirs:
        print(f"[跳过] 未在 {root} 下找到 {SEARCH_DIR_NAME} 目录")
        return pd.DataFrame(
            columns=["root", "artificial_dir", "sample", "table_type", "file"]
        )

    output_records = []

    for artificial_dir in artificial_dirs:
        print(f"处理目录: {artificial_dir}")
        output_dir = make_output_bucket(root, artificial_dir)
        sample_chain_files = collect_chain_files(artificial_dir)

        if not sample_chain_files:
            print(f"  - 未发现符合命名规则的链文件")
            continue

        for sample, chain_files in sorted(sample_chain_files.items()):
            chain_tables: Dict[str, pd.DataFrame] = {}

            for chain in ["TRA", "TRB", "TRG", "TRD"]:
                file_path = chain_files.get(chain)
                if file_path is None:
                    continue

                try:
                    table = load_chain_top_table(file_path, chain=chain, top_n=top_n)
                except Exception as exc:
                    print(f"  - {sample} {chain} 读取失败: {exc}")
                    continue

                chain_tables[chain] = table

                output_file = output_dir / f"{sample}__{chain}_top{top_n}.csv"
                table.to_csv(output_file, index=False, encoding="utf-8-sig")
                output_records.append(
                    {
                        "root": str(root),
                        "artificial_dir": str(artificial_dir),
                        "sample": sample,
                        "table_type": f"single_{chain}",
                        "file": str(output_file),
                    }
                )

        if PAIRED:
            for left_chain, right_chain in PAIRINGS:
                if left_chain not in chain_tables or right_chain not in chain_tables:
                    continue

                pair_table = build_pair_table(
                    chain_tables[left_chain], chain_tables[right_chain]
                )
                pair_file = (
                    output_dir
                    / f"{sample}__{left_chain}_{right_chain}_paired_top{top_n}.csv"
                )
                pair_table.to_csv(pair_file, index=False, encoding="utf-8-sig")
                output_records.append(
                    {
                        "root": str(root),
                        "artificial_dir": str(artificial_dir),
                        "sample": sample,
                        "table_type": f"paired_{left_chain}_{right_chain}",
                        "file": str(pair_file),
                    }
                )

    return pd.DataFrame(output_records)


In [8]:
all_outputs = []

for root_path in INPUT_ROOTS:
    result = process_root(root_path, top_n=TOP_N)
    if not result.empty:
        all_outputs.append(result)

if all_outputs:
    summary_df = pd.concat(all_outputs, ignore_index=True)
    display(summary_df)
    print(f"共生成 {len(summary_df)} 个输出文件")
else:
    summary_df = pd.DataFrame(columns=["root", "artificial_dir", "sample", "table_type", "file"])
    print("没有生成任何输出文件，请检查输入路径和字段名。")


处理目录: /colddata/data_shared260414/To_ZQY/260423_lu/ARP_20250118_reports/mouse ab/SHMYS_LLM_03/artificial_peps
处理目录: /colddata/data_shared260414/To_ZQY/260423_lu/SHMYS_DXP_3/artificial_peps


,root,artificial_dir,sample,table_type,file
0,/colddata/data_shared260414/To_ZQY/260423_lu/A...,/colddata/data_shared260414/To_ZQY/260423_lu/A...,SL_03_CT,single_TRA,/colddata/data_shared260414/To_ZQY/260423_lu/A...
1,/colddata/data_shared260414/To_ZQY/260423_lu/A...,/colddata/data_shared260414/To_ZQY/260423_lu/A...,SL_03_CT,single_TRB,/colddata/data_shared260414/To_ZQY/260423_lu/A...
2,/colddata/data_shared260414/To_ZQY/260423_lu/A...,/colddata/data_shared260414/To_ZQY/260423_lu/A...,SL_03_CT,paired_TRA_TRB,/colddata/data_shared260414/To_ZQY/260423_lu/A...
3,/colddata/data_shared260414/To_ZQY/260423_lu/A...,/colddata/data_shared260414/To_ZQY/260423_lu/A...,SL_03_NH_LLM_CD4C,single_TRA,/colddata/data_shared260414/To_ZQY/260423_lu/A...
4,/colddata/data_shared260414/To_ZQY/260423_lu/A...,/colddata/data_shared260414/To_ZQY/260423_lu/A...,SL_03_NH_LLM_CD4C,single_TRB,/colddata/data_shared260414/To_ZQY/260423_lu/A...
5,/colddata/data_shared260414/To_ZQY/260423_lu/A...,/colddata/data_shared260414/To_ZQY/260423_lu/A...,SL_03_NH_LLM_CD4C,paired_TRA_TRB,/colddata/data_shared260414/To_ZQY/260423_lu/A...
6,/colddata/data_shared260414/To_ZQY/260423_lu/A...,/colddata/data_shared260414/To_ZQY/260423_lu/A...,SL_03_NH_LLM_CD4T,single_TRA,/colddata/data_shared260414/To_ZQY/260423_lu/A...
7,/colddata/data_shared260414/To_ZQY/260423_lu/A...,/colddata/data_shared260414/To_ZQY/260423_lu/A...,SL_03_NH_LLM_CD4T,single_TRB,/colddata/data_shared260414/To_ZQY/260423_lu/A...
8,/colddata/data_shared260414/To_ZQY/260423_lu/A...,/colddata/data_shared260414/To_ZQY/260423_lu/A...,SL_03_NH_LLM_CD4T,paired_TRA_TRB,/colddata/data_shared260414/To_ZQY/260423_lu/A...
9,/colddata/data_shared260414/To_ZQY/260423_lu/A...,/colddata/data_shared260414/To_ZQY/260423_lu/A...,SL_03_NH_LLM_CD8C,single_TRA,/colddata/data_shared260414/To_ZQY/260423_lu/A...


共生成 27 个输出文件


## 使用说明

1. 修改 `INPUT_ROOTS` 为一个或多个待处理的路径。
2. 按需修改 `TOP_N`。
3. 运行全部单元格。
4. 输出目录会生成为：`<输入路径>/artificial_peps_top_clones/`。
5. 如果一个输入路径下存在多个 `artificial_peps` 目录，会在输出目录中按相对路径拆分子目录。

配对表格的组织方式为：

- `index = 1` 先输出 `Alpha/TRA`，再输出同一 `index = 1` 的 `Beta/TRB`
- `TRG/TRD` 配对表格同理
